In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Getting Started with Jina Embeddings v3 on Vertex AI

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/generative_ai/notebook_jina_embeddings_v3_intro.ipynb">
      <img src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fofficial%2Fgenerative_ai%2Fnotebook_jina_embeddings_v3_intro.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/main/notebooks/official/generative_ai/notebook_jina_embeddings_v3_intro.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/main/notebooks/official/generative_ai/notebook_jina_embeddings_v3_intro.ipynb">
      <img src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

## Overview

This notebook demonstrates how to deploy and use **jina-embeddings-v3** from Vertex AI Model Garden.

You will learn how to:
- Deploy jina-embeddings-v3 to a Vertex AI endpoint
- Generate text embeddings using the deployed model
- Use task-specific LoRA adapters for retrieval, classification, and clustering
- Control embedding dimensions with Matryoshka representation
- Compute cross-lingual semantic similarity

### Jina Embeddings v3 on Vertex AI

jina-embeddings-v3 is available as a self-deploy model on Vertex AI Model Garden. You deploy the model to your own Vertex AI endpoint on GPU hardware.

For more information, see the [Jina AI Embedding documentation](https://jina.ai/embeddings).

## Get Started

### Install required packages

In [ ]:
! pip3 install --upgrade --quiet google-cloud-aiplatform numpy

### Restart runtime (Colab only)

To use the newly installed packages, you must restart the runtime on Google Colab.

In [ ]:
import sys

if "google.colab" in sys.modules:

    import IPython

    app = IPython.Application.instance()
    app.kernel.do_shutdown(True)

<div class="alert alert-block alert-warning">
<b>The kernel is going to restart. Wait until it's finished before continuing to the next step.</b>
</div>

### Authenticate your notebook environment (Colab only)

Authenticate your environment on Google Colab.

In [ ]:
import sys

if "google.colab" in sys.modules:

    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information and initialize Vertex AI SDK

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com). Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
PROJECT_ID = "[your-project-id]"  # @param {type:"string"}
LOCATION = "us-central1"  # @param ["us-central1", "us-east1", "us-east4", "us-west1", "us-west4", "northamerica-northeast2", "europe-west1", "europe-west2", "europe-west3", "europe-west4", "europe-west6", "me-central2", "asia-east1", "asia-northeast1", "asia-northeast3", "asia-south1", "asia-southeast1", "asia-southeast2"] {type:"string"}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    raise ValueError("Please set your PROJECT_ID")

ENDPOINT = f"https://{LOCATION}-aiplatform.googleapis.com"
PUBLISHER_NAME = "jinaai"
PUBLISHER_MODEL_NAME = "jina-embeddings-v3"

### Import required libraries

In [ ]:
import json
import time

import numpy as np
from google.cloud import aiplatform

## Deploy Model to Vertex AI

The following cells upload the model from Model Garden, create an endpoint, and deploy the model.

### Initialize Vertex AI SDK

In [ ]:
aiplatform.init(project=PROJECT_ID, location=LOCATION)

### Upload Model

In [ ]:
model = aiplatform.Model.upload(
    display_name="jina-embeddings-v3-" + time.strftime("%Y%m%d-%H%M%S"),
    model_garden_source_model_name=f"publishers/{PUBLISHER_NAME}/models/{PUBLISHER_MODEL_NAME}",
)
print(f"Model resource name: {model.resource_name}")

### Create Endpoint

In [ ]:
my_endpoint = aiplatform.Endpoint.create(
    display_name="jina-embeddings-v3-endpoint-" + time.strftime("%Y%m%d-%H%M%S")
)
print(f"Endpoint resource name: {my_endpoint.resource_name}")

### Deploy Model

Deploy to a `g2-standard-8` machine with a single NVIDIA L4 GPU (recommended). The 570M parameter model fits comfortably in 24 GB VRAM. Higher-end GPUs (A100, H100) are also supported for increased throughput.

In [ ]:
MACHINE_TYPE = "g2-standard-8"  # @param ["g2-standard-8", "a2-highgpu-1g", "a2-ultragpu-1g", "a3-highgpu-1g"] {type: "string"}

ACCELERATOR_MAP = {
    "g2-standard-8": "NVIDIA_L4",
    "a2-highgpu-1g": "NVIDIA_TESLA_A100",
    "a2-ultragpu-1g": "NVIDIA_A100_80GB",
    "a3-highgpu-1g": "NVIDIA_H100_80GB",
}
ACCELERATOR_TYPE = ACCELERATOR_MAP[MACHINE_TYPE]
ACCELERATOR_COUNT = 1

print(f"Machine type: {MACHINE_TYPE}")
print(f"Accelerator: {ACCELERATOR_TYPE}")

In [ ]:
model.deploy(
    endpoint=my_endpoint,
    deployed_model_display_name="jina-embeddings-v3-deployed-"
    + time.strftime("%Y%m%d-%H%M%S"),
    traffic_split={"0": 100},
    machine_type=MACHINE_TYPE,
    accelerator_type=ACCELERATOR_TYPE,
    accelerator_count=ACCELERATOR_COUNT,
    min_replica_count=1,
    max_replica_count=1,
)
print("Model deployed successfully.")

## Generate Embeddings

Send embedding requests to the deployed endpoint using `raw_predict`. The request format follows the OpenAI embeddings API schema.

### Helper functions

In [ ]:
def get_embeddings(texts, task="text-matching", dimensions=1024):
    """Get embeddings from the deployed Vertex AI endpoint."""
    payload = {
        "model": "jina-embeddings-v3",
        "task": task,
        "dimensions": dimensions,
        "input": texts,
    }
    response = my_endpoint.raw_predict(
        body=json.dumps(payload),
        headers={"Content-Type": "application/json"},
    )
    result = json.loads(response.text)
    return np.array([d["embedding"] for d in result["data"]])


def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

### Basic embedding request

Generate embeddings for a list of texts using the `retrieval.query` task adapter.

In [ ]:
payload = {
    "model": "jina-embeddings-v3",
    "task": "retrieval.query",
    "dimensions": 1024,
    "input": [
        "What is machine learning?",
        "How do neural networks work?",
    ],
}

response = my_endpoint.raw_predict(
    body=json.dumps(payload),
    headers={"Content-Type": "application/json"},
)
result = json.loads(response.text)

print(f"Model: {result['model']}")
print(f"Number of embeddings: {len(result['data'])}")
print(f"Embedding dimension: {len(result['data'][0]['embedding'])}")
print(f"Token usage: {result['usage']}")

### Task-specific LoRA adapters

jina-embeddings-v3 includes 5 task-specific LoRA adapters. Selecting the right adapter for your use case improves embedding quality.

| Task | Description | Use case |
|---|---|---|
| `retrieval.query` | Query embeddings for asymmetric search | Search queries |
| `retrieval.passage` | Passage embeddings for asymmetric search | Documents to be searched |
| `separation` | Optimized for clustering and re-ranking | Document clustering |
| `classification` | Optimized for classification tasks | Text categorization |
| `text-matching` | Symmetric similarity | Duplicate detection, STS |

### Asymmetric retrieval with query and passage adapters

For search applications, use `retrieval.query` for queries and `retrieval.passage` for documents.

In [ ]:
query = ["What are the health benefits of green tea?"]

passages = [
    "Green tea contains antioxidants called catechins that may help prevent cell damage. Regular consumption has been linked to improved heart health and reduced risk of certain cancers.",
    "Black tea is fully oxidized, giving it a stronger flavor and darker color compared to green tea. It contains more caffeine than green tea.",
    "The process of making ceramic teapots involves shaping clay on a wheel, followed by drying and firing in a kiln at high temperatures.",
]

query_embedding = get_embeddings(query, task="retrieval.query")
passage_embeddings = get_embeddings(passages, task="retrieval.passage")

print("Query:", query[0])
print("\nPassage similarities:")
for i, passage in enumerate(passages):
    sim = cosine_similarity(query_embedding[0], passage_embeddings[i])
    print(f"  [{sim:.4f}] {passage[:80]}...")

### Cross-lingual semantic similarity

jina-embeddings-v3 supports 30+ trained languages. Texts with similar meaning in different languages produce similar embeddings.

In [ ]:
texts = [
    "The weather is beautiful today.",
    "Das Wetter ist heute wundersch\u00f6n.",
    "Le temps est magnifique aujourd'hui.",
    "\u4eca\u5929\u5929\u6c14\u771f\u597d\u3002",
    "I need to fix my bicycle.",
]

embeddings = get_embeddings(texts, task="text-matching")

labels = ["EN: weather", "DE: weather", "FR: weather", "ZH: weather", "EN: bicycle"]

print("Cross-lingual similarity matrix:")
print(f"{'':>15}", end="")
for label in labels:
    print(f"{label:>15}", end="")
print()

for i, label_i in enumerate(labels):
    print(f"{label_i:>15}", end="")
    for j in range(len(labels)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f"{sim:>15.4f}", end="")
    print()

### Matryoshka embeddings: flexible dimensions

jina-embeddings-v3 supports Matryoshka Representation Learning. You can truncate embeddings to smaller dimensions (32, 64, 128, 256, 512, 768) while retaining most of the performance.

In [ ]:
text_pair = [
    "Artificial intelligence is transforming healthcare.",
    "AI applications in medicine are growing rapidly.",
]

print("Similarity at different embedding dimensions:")
for dim in [32, 64, 128, 256, 512, 1024]:
    embs = get_embeddings(text_pair, task="text-matching", dimensions=dim)
    sim = cosine_similarity(embs[0], embs[1])
    print(f"  dim={dim:>4}: similarity={sim:.4f}")

## Cleaning up

To avoid incurring charges, undeploy the model and delete the endpoint when you are done.

In [ ]:
# Undeploy all models from the endpoint
my_endpoint.undeploy_all()

# Delete the endpoint
my_endpoint.delete()

# Delete the model
model.delete()

print("Resources cleaned up.")